# PyDI Data Integration Workflow: Music

This notebook demonstrates how PyDI is used for end-to-end data integration. We'll work with music datasets to showcase the data integration pipeline from schema and entity matching to data fusion.

## Table of Contents
  - [Datasets](#datasets)
- [Part 1: Schema Matching and Value Normalization](#part-1-schema-matching-and-value-normalization)
- [Part 2: Data Loading and Profiling](#part-2-data-loading-and-profiling)
- [Part 3: Entity Matching](#part-3-entity-matching)
  - [Step 1: Blocking](#step-1-blocking)
  - [Step 2: Blocking Evaluation](#step-2-evaluate-blocking-against-ground-truth)
  - [Step 3: Entity Matching with Comparators](#step-3-entity-matching-with-comparators)
  - [Step 4: Entity Matching Evaluation](#step-4-evaluate-matching-against-ground-truth)
- [Part 4: Data Fusion](#part-4-data-fusion)
  - [Step 1: Define Fusion Strategy](#step-1-define-fusion-strategy)
  - [Step 2: Run Fusion](#step-2-run-fusion)
  - [Step 3: Data Fusion Evaluation](#step-3-evaluate-data-fusion)

## Part 1: Schema Matching and Value Normalization

In [49]:
from pathlib import Path

# Paths relative to this notebook
NOTEBOOK_DIR = Path(".").resolve()
INPUT_DIR = NOTEBOOK_DIR  # Base directory for all input files
DATA_DIR = NOTEBOOK_DIR / "data"
SCHEMA_DIR = NOTEBOOK_DIR / "schemamatching"
OUTPUT_DIR = NOTEBOOK_DIR / "output" / "music"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [50]:
import pandas as pd
import json
from PyDI.schemamatching import SchemaTranslator
from PyDI.normalization import load_normalization_spec

## Step 1: Load Target Schema and Normalization Spec

In [51]:
# Load the JSON Schema (used for both matching and normalization)
with open(SCHEMA_DIR / "target_schema.json") as f:
    target_schema = json.load(f)

# Load NormalizationSpec from the same schema
spec = load_normalization_spec(SCHEMA_DIR / "target_schema.json")

# Clear taxonomy settings (we don't want to use taxonomy normalization here)
for col_name, col_spec in spec.columns.items():
    col_spec.taxonomy_path = None
    col_spec.taxonomy_column = None
    col_spec.taxonomy_mapping_path = None

target_columns = list(spec.columns.keys())

# Create empty target DataFrame for schema matching
df_target = pd.DataFrame(columns=target_columns)
df_target.attrs["dataset_name"] = "target_schema"

# Show column types derived from schema
pd.DataFrame([
    {"column": col, "output_type": col_spec.output_type}
    for col, col_spec in spec.columns.items()
])

,column,output_type
0,id,string
1,name,string
2,artist,string
3,release-date,datetime
4,release-country,string
5,label,string
6,genre,string
7,duration,int


## Step 2: Load Source Datasets

In [52]:
mbrainz = pd.read_csv(DATA_DIR / "csv_with_headers" / "musicbrainz.csv")
mbrainz.attrs["dataset_name"] = "mbrainz"
print(f"Columns: {list(mbrainz.columns)}")
mbrainz.head()

Columns: ['id', 'name', 'artist', 'release-date', 'release-country', 'tracks', 'duration']


,id,name,artist,release-date,release-country,tracks,duration
0,mbrainz_1,Fermats Theorem / Sight Beyond,John B,1996-01-01,United Kingdom of Great Britain and Northern I...,"['Fermats Theorem', 'Sight Beyond']",NaN
1,mbrainz_2,Tempest / Inner Sense,Psychosis,1998-12-14,United Kingdom of Great Britain and Northern I...,"['Tempest', 'Inner Sense']",NaN
2,mbrainz_3,The Sign's Alive,Lypid,2000-09-05,United States of America,"[""The Sign's Alive (original mix)"", ""The Sign'...",NaN
3,mbrainz_4,Surrender,Petalpusher,1999-04-27,United States of America,"['Surrender (Petalpusher original)', ""Surrende...",NaN
4,mbrainz_6,Unreasonable Behaviour,"Garnier, Laurent",2000-07-24,France,"['The Warning', 'City Sphere', 'Forgotten Thou...",27.85 minutes


In [53]:
# Show dataset info
print(f"MusicBrainz: {len(mbrainz)} rows, {len(mbrainz.columns)} columns")
mbrainz.info()

MusicBrainz: 4763 rows, 7 columns
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4763 entries, 0 to 4762
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   id               4763 non-null   object
 1   name             4763 non-null   object
 2   artist           4763 non-null   object
 3   release-date     4450 non-null   object
 4   release-country  4015 non-null   object
 5   tracks           4763 non-null   object
 6   duration         262 non-null    object
dtypes: object(7)
memory usage: 260.6+ KB


In [54]:
lastfm = pd.read_csv(DATA_DIR / "csv_with_headers" / "lastfm.csv")
lastfm.attrs["dataset_name"] = "lastfm"
print(f"Columns: {list(lastfm.columns)}")
lastfm.head()

Columns: ['id', 'name', 'artist', 'tracks', 'duration']


,id,name,artist,tracks,duration
0,lastFM_1,John B - Fermats Theorem / Sight Beyond,John B,"['Fermats Theorem', 'Sight Beyond']",903.0
1,lastFM_2,Tempest / Inner Sense,Psychosis,"['Tempest', 'Inner Sense']",734.0
2,lastFM_4,Petalpusher - Surrender,Petalpusher,"['Surrender (Petalpusher Original)', ""Surrende...",1626.0
3,lastFM_8,in the spirit,R. Trent,"['In The Spirit (The Full Experience)', 'In Th...",1265.0
4,lastFM_11,Come Of Age,S. Vitus Dance,"['Bliss', 'Tunnel Vision', 'Catch The Sun', 'M...",1378.0


In [55]:
discogs = pd.read_csv(DATA_DIR / "csv_with_headers" / "discogs.csv")
discogs.attrs["dataset_name"] = "discogs"
print(f"Columns: {list(discogs.columns)}")
discogs.head()

Columns: ['id', 'name', 'artist', 'release-date', 'release-country', 'label', 'genre', 'tracks']


,id,name,artist,release-date,release-country,label,genre,tracks
0,discogs_3,Fermats Theorem / Sight Beyond,John B,1996-01-01,UK,New Identity Recordings,Electronic,"['Fermats Theorem', 'Sight Beyond']"
1,discogs_4,Tempest / Inner Sense,Psychosis,1998-01-01,UK,Renegade Hardware,Electronic,"['Tempest', 'Inner Sense']"
2,discogs_5,The Sign's Alive,Lypid,2000-09-05,United States of America,Statra Recordings,Electronic,"[""The Sign's Alive (Original Mix)"", ""The Sign'..."
3,discogs_6,Surrender,Petalpusher,1999-04-27,United States of America,Naked Music Recordings,Electronic,"['Surrender (Petalpusher Original)', ""Surrende..."
4,discogs_11,Unreasonable Behaviour,Laurent Garnier,2000-06-01,France,F Communications,Electronic,"['The Warning', 'City Sphere', 'Forgotten Thou..."


# Step 3: LLM-Based Schema Matching

In [56]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

from PyDI.schemamatching.llm_based import LLMBasedSchemaMatcher
load_dotenv()

# Initialize matcher with target schema for better context
matcher = LLMBasedSchemaMatcher(
    chat_model=ChatOpenAI(model="gpt-5"),
    num_rows=40,
    target_schema=target_schema,
)

# Match mbrainz dataset
mbrainz_mapping = matcher.match(mbrainz, df_target)

mbrainz_mapping

[INFO ] PyDI.schemamatching.llm_based - Initialized LLMBasedSchemaMatcher with 40 sample rows
[INFO ] PyDI.schemamatching.llm_based - LLM-based schema matching: mbrainz -> target_schema
[INFO ] httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
[INFO ] PyDI.utils.llm - LLM call: {"timestamp": "2026-02-13T11:06:02.367349Z", "row_index": 0, "attempt": 0, "provider_class": "ChatOpenAI", "model": "gpt-5", "duration_ms": 7486.454010009766, "temperature": 0.0, "max_tokens": null, "usage": {"input_tokens": 8277, "output_tokens": 445, "total_tokens": 8722, "input_token_details": {"audio": 0, "cache_read": 0}, "output_token_details": {"audio": 0, "reasoning": 384}}, "request_messages": [{"type": "system", "content": "You are an expert at aligning table schemas.\n\nIMPORTANT: Source column names may be placeholders or generic identifiers (e.g., Attribute_1, col1, field_a, Column_A). When column names are not descriptive, you MUST focus on analyzing the ACTUAL

,source_dataset,source_column,target_dataset,target_column,score,notes
0,mbrainz,id,target_schema,id,0.95,llm_based_matching
1,mbrainz,name,target_schema,name,0.95,llm_based_matching
2,mbrainz,artist,target_schema,artist,0.95,llm_based_matching
3,mbrainz,release-date,target_schema,release-date,0.95,llm_based_matching
4,mbrainz,release-country,target_schema,release-country,0.95,llm_based_matching
5,mbrainz,duration,target_schema,duration,0.95,llm_based_matching


In [57]:
lastfm_mapping = matcher.match(lastfm, df_target)
lastfm_mapping

[INFO ] PyDI.schemamatching.llm_based - LLM-based schema matching: lastfm -> target_schema
[INFO ] httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
[INFO ] PyDI.utils.llm - LLM call: {"timestamp": "2026-02-13T11:06:15.520330Z", "row_index": 0, "attempt": 0, "provider_class": "ChatOpenAI", "model": "gpt-5", "duration_ms": 13120.136976242065, "temperature": 0.0, "max_tokens": null, "usage": {"input_tokens": 3066, "output_tokens": 557, "total_tokens": 3623, "input_token_details": {"audio": 0, "cache_read": 0}, "output_token_details": {"audio": 0, "reasoning": 512}}, "request_messages": [{"type": "system", "content": "You are an expert at aligning table schemas.\n\nIMPORTANT: Source column names may be placeholders or generic identifiers (e.g., Attribute_1, col1, field_a, Column_A). When column names are not descriptive, you MUST focus on analyzing the ACTUAL DATA VALUES to determine semantic meaning:\n- Look at the format, patterns, and content of va

,source_dataset,source_column,target_dataset,target_column,score,notes
0,lastfm,id,target_schema,id,0.95,llm_based_matching
1,lastfm,name,target_schema,name,0.95,llm_based_matching
2,lastfm,artist,target_schema,artist,0.95,llm_based_matching
3,lastfm,duration,target_schema,duration,0.95,llm_based_matching


In [58]:
discogs_mapping = matcher.match(discogs, df_target)
discogs_mapping

[INFO ] PyDI.schemamatching.llm_based - LLM-based schema matching: discogs -> target_schema
[INFO ] httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
[INFO ] PyDI.utils.llm - LLM call: {"timestamp": "2026-02-13T11:06:21.592432Z", "row_index": 0, "attempt": 0, "provider_class": "ChatOpenAI", "model": "gpt-5", "duration_ms": 6001.734256744385, "temperature": 0.0, "max_tokens": null, "usage": {"input_tokens": 3911, "output_tokens": 387, "total_tokens": 4298, "input_token_details": {"audio": 0, "cache_read": 0}, "output_token_details": {"audio": 0, "reasoning": 320}}, "request_messages": [{"type": "system", "content": "You are an expert at aligning table schemas.\n\nIMPORTANT: Source column names may be placeholders or generic identifiers (e.g., Attribute_1, col1, field_a, Column_A). When column names are not descriptive, you MUST focus on analyzing the ACTUAL DATA VALUES to determine semantic meaning:\n- Look at the format, patterns, and content of va

,source_dataset,source_column,target_dataset,target_column,score,notes
0,discogs,id,target_schema,id,0.95,llm_based_matching
1,discogs,name,target_schema,name,0.95,llm_based_matching
2,discogs,artist,target_schema,artist,0.95,llm_based_matching
3,discogs,release-date,target_schema,release-date,0.95,llm_based_matching
4,discogs,release-country,target_schema,release-country,0.95,llm_based_matching
5,discogs,label,target_schema,label,0.95,llm_based_matching
6,discogs,genre,target_schema,genre,0.95,llm_based_matching


## Step 4: Translate and Normalize


In [59]:
# Handle tracks column - parse XML nested content if needed
import ast

def parse_tracks(x):
    """Parse tracks from various formats."""
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return x
    x_str = str(x).strip()
    if not x_str or x_str in ["\n\t\t\t", "None"]:
        return []
    try:
        return ast.literal_eval(x_str)
    except:
        # Try splitting by newlines or commas
        tracks = [t.strip() for t in x_str.replace("\n", ",").replace("\t", "").split(",") if t.strip()]
        return tracks if tracks else []

for df in [mbrainz, lastfm, discogs]:
    if "tracks" in df.columns:
        df["tracks"] = df["tracks"].apply(parse_tracks)

In [60]:
# Ensure duration is numeric
for df in [mbrainz, lastfm, discogs]:
    if "duration" in df.columns:
        df["duration"] = pd.to_numeric(df["duration"], errors="coerce")

In [61]:
# Verify data types
print("MusicBrainz dtypes:")
print(mbrainz.dtypes)

MusicBrainz dtypes:
id                  object
name                object
artist              object
release-date        object
release-country     object
tracks              object
duration           float64
dtype: object


In [62]:
# Show sample of cleaned data
print("Sample MusicBrainz record:")
print(mbrainz.iloc[0])

Sample MusicBrainz record:
id                                                         mbrainz_1
name                                  Fermats Theorem / Sight Beyond
artist                                                        John B
release-date                                              1996-01-01
release-country    United Kingdom of Great Britain and Northern I...
tracks                               [Fermats Theorem, Sight Beyond]
duration                                                         NaN
Name: 0, dtype: object


In [63]:
# Verify all datasets have id column
print(f"MusicBrainz has id: {'id' in mbrainz.columns}")
print(f"LastFM has id: {'id' in lastfm.columns}")
print(f"Discogs has id: {'id' in discogs.columns}")

MusicBrainz has id: True
LastFM has id: True
Discogs has id: True


In [64]:
translator = SchemaTranslator()

# Translate + normalize each dataset with its own mapping

spec.set_column("release-country", country_format="name")
spec.set_column("release-date", output_type="datetime")

discogs_normalized = translator.translate(
    discogs, discogs_mapping,
    normalize =spec, on_failure="keep"
)

lastfm_normalized = translator.translate(
    lastfm, lastfm_mapping,
    normalize=spec, on_failure="keep"
)

mbrainz_normalized = translator.translate(
    mbrainz, mbrainz_mapping,
    normalize=spec, on_failure="keep"
)

[INFO ] root - Translating 7 columns for 'discogs'
[WARNING] PyDI.normalization.transform - Column 'duration' not found in DataFrame
[INFO ] root - Normalization complete: 41041 values transformed, 1379 values failed
[INFO ] root - Translating 4 columns for 'lastfm'
[WARNING] PyDI.normalization.transform - Column 'release-date' not found in DataFrame
[WARNING] PyDI.normalization.transform - Column 'release-country' not found in DataFrame
[WARNING] PyDI.normalization.transform - Column 'label' not found in DataFrame
[WARNING] PyDI.normalization.transform - Column 'genre' not found in DataFrame
[INFO ] root - Normalization complete: 4635 values transformed, 0 values failed
[INFO ] root - Translating 6 columns for 'mbrainz'
[WARNING] PyDI.normalization.transform - Column 'label' not found in DataFrame
[WARNING] PyDI.normalization.transform - Column 'genre' not found in DataFrame
[INFO ] root - Normalization complete: 8464 values transformed, 1 values failed


In [65]:
# Inspect normalized mbrainz dataset (target columns only)
mbrainz_cols = [c for c in target_columns if c in mbrainz_normalized.columns]
mbrainz_normalized[mbrainz_cols].head(10)

,id,name,artist,release-date,release-country,duration
0,mbrainz_1,Fermats Theorem / Sight Beyond,John B,1996-01-01,United Kingdom,NaN
1,mbrainz_2,Tempest / Inner Sense,Psychosis,1998-12-14,United Kingdom,NaN
2,mbrainz_3,The Sign's Alive,Lypid,2000-09-05,United States,NaN
3,mbrainz_4,Surrender,Petalpusher,1999-04-27,United States,NaN
4,mbrainz_6,Unreasonable Behaviour,"Garnier, Laurent",2000-07-24,France,NaN
5,mbrainz_7,In the Spirit,"Trent, Ron",1999-01-01,United Kingdom,NaN
6,mbrainz_8,The Dance,"Gallery Collective, The",1996-01-01,United States,NaN
7,mbrainz_9,Come of Age,St. Vitus Dance,1994-01-01,United Kingdom,NaN
8,mbrainz_10,Contrax / All Mighty,Decorum,1999-01-01,United Kingdom,NaN
9,mbrainz_11,Electronically Tested,Surgeon,1995-01-01,United Kingdom,NaN


In [66]:
discogs_cols = [c for c in target_columns if c in discogs_normalized.columns]
discogs_normalized[discogs_cols].head(10)

,id,name,artist,release-date,release-country,label,genre
0,discogs_3,Fermats Theorem / Sight Beyond,John B,1996-01-01,Uganda,New Identity Recordings,Electronic
1,discogs_4,Tempest / Inner Sense,Psychosis,1998-01-01,Uganda,Renegade Hardware,Electronic
2,discogs_5,The Sign's Alive,Lypid,2000-09-05,United States,Statra Recordings,Electronic
3,discogs_6,Surrender,Petalpusher,1999-04-27,United States,Naked Music Recordings,Electronic
4,discogs_11,Unreasonable Behaviour,Laurent Garnier,2000-06-01,France,F Communications,Electronic
5,discogs_13,In The Spirit,Ron Trent,1999-01-01,Uganda,Peacefrog Records,Electronic
6,discogs_14,The Dance,The Gallery Collective,1996-01-01,United States,Prescription,Electronic
7,discogs_16,Analogue,Mampi Swift,1997-01-01,Uganda,Suburban Base Records,Electronic
8,discogs_17,Come Of Age,St. Vitus Dance,1994-01-01,Uganda,Peacefrog Records,Electronic
9,discogs_18,Ebony Angel - The Resurrection,Monica Elam,1999-01-01,United States,Clairaudience,Electronic


In [67]:
lastfm_cols = [c for c in target_columns if c in lastfm_normalized.columns]
lastfm_normalized[lastfm_cols].head(10)

,id,name,artist,duration
0,lastFM_1,John B - Fermats Theorem / Sight Beyond,John B,903.0
1,lastFM_2,Tempest / Inner Sense,Psychosis,734.0
2,lastFM_4,Petalpusher - Surrender,Petalpusher,1626.0
3,lastFM_8,in the spirit,R. Trent,1265.0
4,lastFM_11,Come Of Age,S. Vitus Dance,1378.0
5,lastFM_16,Devotional,D. Alvarado,944.0
6,lastFM_24,P. Johnson - The Music In Me,P. Johnson,1168.0
7,lastFM_27,On and,GH-106,120.0
8,lastFM_28,- new - Subtle Frequencies,C. Jackson,613.0
9,lastFM_30,Second Area / Think Tank,Inigo Kennedy,827.0


In [68]:
# Only keep target columns
mbrainz = mbrainz_normalized[mbrainz_cols].copy()
lastfm = lastfm_normalized[lastfm_cols].copy()
discogs = discogs_normalized[discogs_cols].copy()

In [69]:
# Sum up track duration lists into total album duration
def sum_track_durations(durations):
    if isinstance(durations, list):
        vals = [int(d) for d in durations if isinstance(d, (int, float, np.integer, np.floating))]
        return sum(vals) if vals else pd.NA
    return pd.NA

# for discogs and mbrainz put scalars within list attributes into lists
def wrap_in_list_if_scalar(x):
    if isinstance(x, list):
        return x
    elif pd.isna(x):
        return x
    else:
        return [x]

#for col in ["tracks_track_name", "tracks_track_duration", "tracks_track_position"]:
#    discogs[col] = discogs[col].apply(wrap_in_list_if_scalar)
#    mbrainz[col] = mbrainz[col].apply(wrap_in_list_if_scalar)

#lastfm["duration"] = lastfm["tracks_track_duration"].apply(sum_track_durations)
#discogs["duration"] = discogs["tracks_track_duration"].apply(sum_track_durations)
#mbrainz["duration"] = mbrainz["tracks_track_duration"].apply(sum_track_durations)

## Part 2: Data Loading and Profiling

In [70]:
# Display basic information
datasets = [discogs, mbrainz, lastfm]
names = ["Discogs", "MusicBrainz", "Last.fm"]

total_records = sum(len(df) for df in datasets)
print(f"Total records across all datasets: {total_records:,}")

Total records across all datasets: 37,255


In [71]:
from PyDI.utils import DataProfiler

# Initialize the DataProfiler
profiler = DataProfiler()

for df, name in zip(datasets, names):
    profile = profiler.summary(df) # automatically prints some statistics and returns object containing stats

display(profile)

discogs:
  Rows: 22,627
  Columns: 7
  Total nulls: 2,834
  Null percentage: 1.8%
  Null counts per column:
    release-date: 2,234 (9.9%)
    release-country: 600 (2.7%)

mbrainz:
  Rows: 4,763
  Columns: 6
  Total nulls: 5,824
  Null percentage: 20.4%
  Null counts per column:
    release-date: 313 (6.6%)
    release-country: 748 (15.7%)
    duration: 4,763 (100.0%)

lastfm:
  Rows: 9,865
  Columns: 4
  Total nulls: 5,230
  Null percentage: 13.3%
  Null counts per column:
    duration: 5,230 (53.0%)



{'rows': 9865,
 'columns': 4,
 'nulls_total': 5230,
 'nulls_per_column': {'id': 0, 'name': 0, 'artist': 0, 'duration': 5230},
 'dtypes': {'id': 'object',
  'name': 'object',
  'artist': 'object',
  'duration': 'float64'}}

### Attribute Coverage Analysis

In [72]:
coverage = profiler.analyze_coverage(
    datasets=datasets,
    include_samples=True,
    sample_count=3  # Show 3 sample values per attribute
)

print("📊 Attribute coverage across datasets:")
display(coverage)

# Identify attributes suitable for entity matching
print("\n🔗 Attributes suitable for entity matching:")
matching_attrs = coverage[coverage['datasets_with_attribute'] >= 2]['attribute'].tolist()
print(f"Attributes available in 2+ datasets: {matching_attrs}")

[INFO ] PyDI.fusion.analysis - Analyzed 8 attributes across 3 datasets


📊 Attribute coverage across datasets:


,attribute,discogs_count,discogs_pct,discogs_coverage,discogs_samples,mbrainz_count,mbrainz_pct,mbrainz_coverage,mbrainz_samples,lastfm_count,lastfm_pct,lastfm_coverage,lastfm_samples,avg_coverage,max_coverage,datasets_with_attribute
0,artist,22627/22627,100.0%,1.000000,"['John B', 'Psychosis', 'Lypid']",4763/4763,100.0%,1.000000,"['John B', 'Psychosis', 'Lypid']",9865/9865,100.0%,1.000000,"['John B', 'Psychosis', 'Petalpusher']",1.000000,1.000000,3
1,duration,0/0,0%,0.000000,N/A,0/4763,0.0%,0.000000,[],4635/9865,47.0%,0.469843,"[903.0, 734.0, 1626.0]",0.156614,0.469843,1
2,genre,22627/22627,100.0%,1.000000,"['Electronic', 'Electronic', 'Electronic']",0/0,0%,0.000000,N/A,0/0,0%,0.000000,N/A,0.333333,1.000000,1
3,id,22627/22627,100.0%,1.000000,"['discogs_3', 'discogs_4', 'discogs_5']",4763/4763,100.0%,1.000000,"['mbrainz_1', 'mbrainz_2', 'mbrainz_3']",9865/9865,100.0%,1.000000,"['lastFM_1', 'lastFM_2', 'lastFM_4']",1.000000,1.000000,3
4,label,22627/22627,100.0%,1.000000,"['New Identity Recordings', 'Renegade Hardware...",0/0,0%,0.000000,N/A,0/0,0%,0.000000,N/A,0.333333,1.000000,1
5,name,22627/22627,100.0%,1.000000,"['Fermats Theorem / Sight Beyond', 'Tempest / ...",4763/4763,100.0%,1.000000,"['Fermats Theorem / Sight Beyond', 'Tempest / ...",9865/9865,100.0%,1.000000,"['John B - Fermats Theorem / Sight Beyond', '...",1.000000,1.000000,3
6,release-country,22027/22627,97.3%,0.973483,"['Uganda', 'Uganda', 'United States']",4015/4763,84.3%,0.842956,"['United Kingdom', 'United Kingdom', 'United S...",0/0,0%,0.000000,N/A,0.605480,0.973483,2
7,release-date,20393/22627,90.1%,0.901268,"[Timestamp('1996-01-01 00:00:00'), Timestamp('...",4450/4763,93.4%,0.934285,"[Timestamp('1996-01-01 00:00:00'), Timestamp('...",0/0,0%,0.000000,N/A,0.611851,0.934285,2



🔗 Attributes suitable for entity matching:
Attributes available in 2+ datasets: ['artist', 'id', 'name', 'release-country', 'release-date']


### Detailed Data Profiling

In [73]:
from pathlib import Path

# Generate detailed HTML profiles for each dataset
profile_dir = OUTPUT_DIR / "dataset-profiles"
profile_dir.mkdir(parents=True, exist_ok=True)

profile_paths = []

for df, name in zip(datasets, names):
    print(f"Profiling {name}...")
    
    profile_path = profiler.profile(df, str(profile_dir))
    profile_paths.append(profile_path)
    print(f"Profile saved: {profile_path}")

print(f"\n Generated {len(profile_paths)} detailed HTML reports")
print(f" Location: {profile_dir}")
print("\n Open these HTML files in your browser for interactive exploration:")
for path in profile_paths:
    print(f"  • {Path(path).name}")


Profiling Discogs...


Export report to file: 100%|██████████| 1/1 [00:00<00:00, 495.60it/s]


Profile saved: /Users/aaronsteiner/Documents/GitHub/unsupervised-data-integration/usecases/input/music/output/music/dataset-profiles/discogs_profile.html
Profiling MusicBrainz...


Export report to file: 100%|██████████| 1/1 [00:00<00:00, 379.64it/s]


Profile saved: /Users/aaronsteiner/Documents/GitHub/unsupervised-data-integration/usecases/input/music/output/music/dataset-profiles/mbrainz_profile.html
Profiling Last.fm...


Export report to file: 100%|██████████| 1/1 [00:00<00:00, 245.04it/s]

Profile saved: /Users/aaronsteiner/Documents/GitHub/unsupervised-data-integration/usecases/input/music/output/music/dataset-profiles/lastfm_profile.html

 Generated 3 detailed HTML reports
 Location: /Users/aaronsteiner/Documents/GitHub/unsupervised-data-integration/usecases/input/music/output/music/dataset-profiles

 Open these HTML files in your browser for interactive exploration:
  • discogs_profile.html
  • mbrainz_profile.html
  • lastfm_profile.html


## Part 3: Entity Matching

### Step 1: Blocking

In [74]:
# Set up logging
import logging

import os
os.makedirs('output/logs', exist_ok=True)

logging.basicConfig(
    level=logging.INFO, # Alternatively, use logging.DEBUG for more verbosity
    format='[%(levelname)-5s] %(name)s - %(message)s',
    handlers=[
          logging.FileHandler('output/logs/pydi.log'),  # Save to file
          logging.StreamHandler()                      # Display on console
      ],
    force=True
)

In [75]:
# Import blocking methods
from PyDI.entitymatching import StandardBlocker, SortedNeighbourhoodBlocker, TokenBlocker, EmbeddingBlocker
import re

# Standard Blocking - Longest Token in Name
# Add name_longest_token directly to the original dataframes
def get_longest_token(name):
    tokens = re.split(r"[^A-Za-z0-9_']+", str(name))    
    tokens = [t for t in tokens if t]
    return max(tokens, key=len) if tokens else ''

mbrainz['name_longest_token'] = mbrainz['name'].apply(get_longest_token)
discogs['name_longest_token'] = discogs['name'].apply(get_longest_token)
lastfm['name_longest_token'] = lastfm['name'].apply(get_longest_token)

standard_blocker_m2d = StandardBlocker(
    mbrainz, discogs,
    on=['name_longest_token'],
    batch_size=1000,
    output_dir=OUTPUT_DIR / "blocking-evaluation",
    id_column='id'
 )

standard_blocker_m2l = StandardBlocker(
    mbrainz, lastfm,
    on=['name_longest_token'],
    batch_size=1000,
    output_dir=OUTPUT_DIR / "blocking-evaluation",
    id_column='id'
 )

[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 1720 blocking keys for first dataset
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 4685 blocking keys for second dataset
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 1443 blocks from blocking keys
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - Debug results written to file: /Users/aaronsteiner/Documents/GitHub/unsupervised-data-integration/usecases/input/music/output/music/blocking-evaluation/debugResultsBlocking_StandardBlocker.csv
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 1720 blocking keys for first dataset
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 2850 blocking keys for second dataset
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 1368 blocks from blocking keys
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - Debug results written to file: /Use

### Step 2: Evaluate Blocking Against Ground Truth

In [76]:
import pandas as pd
from PyDI.io import load_csv
from PyDI.entitymatching import EntityMatchingEvaluator
# Showcase EntityMatchingEvaluator.evaluate_blocking utility

# Load test set with proper column names
test_gt = load_csv(
    INPUT_DIR / "entitymatching" / "musicbrainz_2_discogs_val.csv",
    name="test_set", header=None, names=['id1', 'id2', 'label'], add_index=False
)

# Use EntityMatchingEvaluator.evaluate_blocking_batched on Standard Blocking
results = EntityMatchingEvaluator.evaluate_blocking_batched(
    blocker=standard_blocker_m2d,
    test_pairs=test_gt,
    out_dir=OUTPUT_DIR / "blocking-evaluation"
)

display(results)

[INFO ] root - Starting batched blocking evaluation...
[INFO ] root - Processed 10 batches, 10000 pairs, 10 true matches
[INFO ] root - Processed 20 batches, 20000 pairs, 13 true matches
[INFO ] root - Processed 30 batches, 30000 pairs, 25 true matches
[INFO ] root - Processed 40 batches, 40000 pairs, 42 true matches
[INFO ] root - Processed 50 batches, 50000 pairs, 48 true matches
[INFO ] root - Processed 60 batches, 60000 pairs, 55 true matches
[INFO ] root - Processed 70 batches, 70000 pairs, 60 true matches
[INFO ] root - Processed 80 batches, 80000 pairs, 63 true matches
[INFO ] root - Processed 90 batches, 90000 pairs, 79 true matches
[INFO ] root - Processed 100 batches, 100000 pairs, 98 true matches
[INFO ] root - Processed 110 batches, 110000 pairs, 110 true matches
[INFO ] root - Processed 120 batches, 120000 pairs, 131 true matches
[INFO ] root - Processed 130 batches, 130000 pairs, 137 true matches
[INFO ] root - Processed 140 batches, 140000 pairs, 141 true matches
[INFO ]

{'pair_completeness': 1.0,
 'pair_quality': 0.002145844289888658,
 'reduction_ratio': 0.9967007230357613,
 'total_candidates': 355571,
 'total_possible_pairs': 107772401,
 'true_positives_found': 763,
 'total_true_pairs': 763,
 'batches_processed': 356,
 'evaluation_timestamp': '2026-02-13T12:06:52.833049',
 'output_files': ['/Users/aaronsteiner/Documents/GitHub/unsupervised-data-integration/usecases/input/music/output/music/blocking-evaluation/blocking_evaluation_summary.json',
  '/Users/aaronsteiner/Documents/GitHub/unsupervised-data-integration/usecases/input/music/output/music/blocking-evaluation/blocking_detailed_results.csv']}

In [77]:
# Now evaluate blocking for musicbrainz and lastfm combination

# Load test set with proper column names
test_gt = load_csv(
    INPUT_DIR / "entitymatching" / "musicbrainz_2_lastfm_val.csv",
    name="test_set", header=None, names=['id1', 'id2', 'label'], add_index=False
)

# Use EntityMatchingEvaluator.evaluate_blocking_batched on Standard Blocking
results = EntityMatchingEvaluator.evaluate_blocking_batched(
    blocker=standard_blocker_m2l,
    test_pairs=test_gt,
    out_dir=OUTPUT_DIR / "blocking-evaluation"
)

display(results)

[INFO ] root - Starting batched blocking evaluation...
[INFO ] root - Processed 10 batches, 10000 pairs, 17 true matches
[INFO ] root - Processed 20 batches, 20000 pairs, 40 true matches
[INFO ] root - Processed 30 batches, 30000 pairs, 51 true matches
[INFO ] root - Processed 40 batches, 40000 pairs, 65 true matches
[INFO ] root - Processed 50 batches, 50000 pairs, 93 true matches
[INFO ] root - Processed 60 batches, 60000 pairs, 114 true matches
[INFO ] root - Processed 70 batches, 70000 pairs, 143 true matches
[INFO ] root - Processed 80 batches, 80000 pairs, 181 true matches
[INFO ] root - Processed 90 batches, 90000 pairs, 202 true matches
[INFO ] root - Processed 100 batches, 100000 pairs, 238 true matches
[INFO ] root - Processed 110 batches, 110000 pairs, 269 true matches
[INFO ] root - Processed 120 batches, 120000 pairs, 337 true matches
[INFO ] root - Processed 130 batches, 130000 pairs, 377 true matches
[INFO ] root - Processed 140 batches, 140000 pairs, 406 true matches
[I

{'pair_completeness': 0.9292682926829269,
 'pair_quality': 0.004941345835846157,
 'reduction_ratio': 0.9967180493240736,
 'total_candidates': 154209,
 'total_possible_pairs': 46986995,
 'true_positives_found': 762,
 'total_true_pairs': 820,
 'batches_processed': 155,
 'evaluation_timestamp': '2026-02-13T12:06:56.748063',
 'output_files': ['/Users/aaronsteiner/Documents/GitHub/unsupervised-data-integration/usecases/input/music/output/music/blocking-evaluation/blocking_evaluation_summary.json',
  '/Users/aaronsteiner/Documents/GitHub/unsupervised-data-integration/usecases/input/music/output/music/blocking-evaluation/blocking_detailed_results.csv']}

### Step 3: Entity Matching with Comparators

In [78]:
from PyDI.entitymatching import StringComparator, DateComparator, NumericComparator

# ignore case and punctuation
def normalize_text(s: str) -> str: 
    if s is None:
        return ""
    return re.sub(r"[^\w\s]|_", "", s).lower()

comparators = [
    # Release name — Jaccard
    StringComparator(
        column='name', 
        similarity_function='jaccard',
        preprocess=normalize_text
    ),
    # Release artist — Jaccard
    StringComparator(
        column='artist',
        similarity_function='jaccard',
        preprocess=normalize_text
    ),
    # Release duration — within 10% --> allow 10% deviation
   
    # Release date — within 2 years
    DateComparator(
        column='release-date',
        max_days_difference=365 * 2
    ),
    # Release country — Jaccard
    StringComparator(
        column='release-country',
        similarity_function='jaccard',
        preprocess=normalize_text
    ),
]

Next, we setup the matcher and run the matching with our chosen best blocking method:

In [79]:
import numpy as np

# Convert lists in mbrainz["duration"] to single integer values (sum if list, else int)

def sum_duration(val):
    if isinstance(val, list):
        return int(np.nansum([int(x) for x in val if str(x).isdigit()]))
    try:
        return int(val)
    except Exception:
        return np.nan

mbrainz["duration"] = mbrainz["duration"].apply(sum_duration)

In [80]:
from PyDI.entitymatching import RuleBasedMatcher

# Initialize Rule-Based Matcher
matcher = RuleBasedMatcher()

correspondences_m2d = matcher.match(
    df_left=mbrainz,
    df_right=discogs, 
    candidates=standard_blocker_m2d, # pass the blocker, which will internally generate candidate pairs using batching
    comparators=comparators,
    weights=None, # equal weights for all 4 comparators
    threshold=0.5,
    id_column='id'
)

[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Starting Entity Matching
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Blocking 4763 x 22627 elements
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Matching 4763 x 22627 elements after 0:00:0.062; 355571 blocked pairs (reduction ratio: 0.9967007230357613)
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Entity Matching finished after 0:00:33.405; found 6247 correspondences.


In [81]:
matcher = RuleBasedMatcher()

# Remove release-date and release-country comparator since those columns are not present in lastfm dataset
comparators.pop()
comparators.pop()

correspondences_m2l = matcher.match(
    df_left=mbrainz,
    df_right=lastfm, 
    candidates=standard_blocker_m2l,
    comparators=comparators,
    weights=None, # equal weights for all 4 comparators
    threshold=0.3,
    id_column='id'
)

[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Starting Entity Matching
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Blocking 4763 x 9865 elements
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Matching 4763 x 9865 elements after 0:00:0.028; 154209 blocked pairs (reduction ratio: 0.9967180493240736)
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Entity Matching finished after 0:00:11.727; found 14370 correspondences.


### Step 4: Evaluate Matching Against Ground Truth

In [82]:
gt_test = load_csv(
    INPUT_DIR / "entitymatching" / "musicbrainz_2_discogs_test.csv", 
    name="test_entity_matching",
    header=None,
    names=['id1', 'id2', 'label'],
    add_index=False
)

debug_output_dir = OUTPUT_DIR / "debug_results_entity_matching"
debug_output_dir.mkdir(parents=True, exist_ok=True)

eval_results = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_m2d,
    test_pairs=gt_test,
    out_dir=debug_output_dir
)

display(eval_results)

[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  329
[INFO ] root -   True Negatives:  622
[INFO ] root -   False Positives: 45
[INFO ] root -   False Negatives: 4
[INFO ] root - Performance Metrics:
[INFO ] root -   Accuracy:  0.951
[INFO ] root -   Precision: 0.880
[INFO ] root -   Recall:    0.988
[INFO ] root -   F1-Score:  0.931


{'precision': 0.8796791443850267,
 'recall': 0.987987987987988,
 'f1': 0.9306930693069306,
 'accuracy': 0.951,
 'true_positives': 329,
 'false_positives': 45,
 'false_negatives': 4,
 'true_negatives': 622,
 'threshold_used': 0.0,
 'total_correspondences': 6247,
 'filtered_correspondences': 6247,
 'evaluation_timestamp': '2026-02-13T12:07:43.618021',
 'output_files': ['/Users/aaronsteiner/Documents/GitHub/unsupervised-data-integration/usecases/input/music/output/music/debug_results_entity_matching/matching_evaluation_summary.json',
  '/Users/aaronsteiner/Documents/GitHub/unsupervised-data-integration/usecases/input/music/output/music/debug_results_entity_matching/matching_detailed_results.csv']}

In [83]:
print("Analyzing cluster size distribution in our entity matching results...")

# Create cluster size distribution from our matches
cluster_distribution = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_m2d,
    out_dir=str(OUTPUT_DIR / "cluster_analysis")
)

print(f"\n📊 Cluster Size Distribution Results:")
display(cluster_distribution)

[INFO ] PyDI.entitymatching.evaluation - Cluster Size Distribution of 2791 clusters:
[INFO ] PyDI.entitymatching.evaluation - 	Cluster Size	| Frequency	| Percentage
[INFO ] PyDI.entitymatching.evaluation - 	──────────────────────────────────────────────────
[INFO ] PyDI.entitymatching.evaluation - 		2	|	1972	|	70.66%
[INFO ] PyDI.entitymatching.evaluation - 		3	|	397	|	14.22%
[INFO ] PyDI.entitymatching.evaluation - 		4	|	178	|	6.38%
[INFO ] PyDI.entitymatching.evaluation - 		5	|	74	|	2.65%
[INFO ] PyDI.entitymatching.evaluation - 		6	|	55	|	1.97%
[INFO ] PyDI.entitymatching.evaluation - 		7	|	20	|	0.72%
[INFO ] PyDI.entitymatching.evaluation - 		8	|	19	|	0.68%
[INFO ] PyDI.entitymatching.evaluation - 		9	|	16	|	0.57%
[INFO ] PyDI.entitymatching.evaluation - 		10	|	10	|	0.36%
[INFO ] PyDI.entitymatching.evaluation - 		11	|	2	|	0.07%
[INFO ] PyDI.entitymatching.evaluation - 		12	|	5	|	0.18%
[INFO ] PyDI.entitymatching.evaluation - 		13	|	3	|	0.11%
[INFO ] PyDI.entitymatching.evaluation 

Analyzing cluster size distribution in our entity matching results...

📊 Cluster Size Distribution Results:


,cluster_size,frequency,percentage
0,2,1972,70.655679
1,3,397,14.224292
2,4,178,6.377642
3,5,74,2.651379
4,6,55,1.970620
5,7,20,0.716589
6,8,19,0.680760
7,9,16,0.573271
8,10,10,0.358295
9,11,2,0.071659


In [84]:
# Write out detailed cluster information with all entity records for debugging purposes

# Use the matches we found earlier to demonstrate cluster details
cluster_details_path = OUTPUT_DIR / "cluster_analysis" / "detailed_cluster_info.json"

# Call write_cluster_details with our entity matches
output_path = EntityMatchingEvaluator.write_cluster_details(
    correspondences=correspondences_m2d,
    out_path=cluster_details_path
)

[INFO ] root - Cluster details written to /Users/aaronsteiner/Documents/GitHub/unsupervised-data-integration/usecases/input/music/output/music/cluster_analysis/detailed_cluster_info.json
[INFO ] root - Exported 2791 clusters with detailed record information


Additionally, PyDI offers 6 different post-clustering methods to "clean" clusters after entity matching. For example, if we want to enforce that each record in a dataset can only have exactly one correspondence in the other dataset, we can apply a greedy one-to-one matching, maximum bipartite matching or stable marriage matching.

In [85]:
from PyDI.entitymatching import MaximumBipartiteMatching

# use Maximum Bipartite Matching to refine results to 1:1 matches
clusterer = MaximumBipartiteMatching()
correspondences_m2d = clusterer.cluster(correspondences_m2d)

cluster_distribution = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_m2d,
    out_dir=str(OUTPUT_DIR / "cluster_analysis")
)

eval_results = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_m2d,
    test_pairs=gt_test,
    out_dir=debug_output_dir
)

[INFO ] root - Filtered correspondences: 6247 -> 6247 (threshold=0.0)
[INFO ] root - Maximum bipartite matching: 6247 -> 3292 
[INFO ] root - MaximumBipartiteMatching: 6247 -> 3292 correspondences
[INFO ] root - MaximumBipartiteMatching: 8245 -> 6584 entities
[INFO ] PyDI.entitymatching.evaluation - Cluster Size Distribution of 3292 clusters:
[INFO ] PyDI.entitymatching.evaluation - 	Cluster Size	| Frequency	| Percentage
[INFO ] PyDI.entitymatching.evaluation - 	──────────────────────────────────────────────────
[INFO ] PyDI.entitymatching.evaluation - 		2	|	3292	|	100.00%
[INFO ] root - Cluster size distribution written to /Users/aaronsteiner/Documents/GitHub/unsupervised-data-integration/usecases/input/music/output/music/cluster_analysis/cluster_size_distribution.csv
[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  249
[INFO ] root -   True Negatives:  656
[INFO ] root -   False Positives: 11
[INFO ] root -   False Negatives: 84
[INFO ] root - Performance Metrics:
[

In [86]:
from PyDI.entitymatching import  MaximumBipartiteMatching

gt_test = load_csv(
    INPUT_DIR / "entitymatching" / "musicbrainz_2_lastfm_test.csv", 
    name="test_entity_matching",
    header=None,
    names=['id1', 'id2', 'label'],
    add_index=False
)

debug_output_dir = OUTPUT_DIR / "debug_results_entity_matching"
debug_output_dir.mkdir(parents=True, exist_ok=True)

eval_results = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_m2l,
    test_pairs=gt_test,
    out_dir=debug_output_dir
)

cluster_distribution = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_m2l,
)

clusterer = MaximumBipartiteMatching()
correspondences_m2l = clusterer.cluster(correspondences_m2l)


cluster_distribution = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_m2l,
)

eval_results = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_m2l,
    test_pairs=gt_test,
    out_dir=debug_output_dir
)

[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  309
[INFO ] root -   True Negatives:  534
[INFO ] root -   False Positives: 133
[INFO ] root -   False Negatives: 24
[INFO ] root - Performance Metrics:
[INFO ] root -   Accuracy:  0.843
[INFO ] root -   Precision: 0.699
[INFO ] root -   Recall:    0.928
[INFO ] root -   F1-Score:  0.797
[INFO ] PyDI.entitymatching.evaluation - Cluster Size Distribution of 2318 clusters:
[INFO ] PyDI.entitymatching.evaluation - 	Cluster Size	| Frequency	| Percentage
[INFO ] PyDI.entitymatching.evaluation - 	──────────────────────────────────────────────────
[INFO ] PyDI.entitymatching.evaluation - 		2	|	1758	|	75.84%
[INFO ] PyDI.entitymatching.evaluation - 		3	|	178	|	7.68%
[INFO ] PyDI.entitymatching.evaluation - 		4	|	93	|	4.01%
[INFO ] PyDI.entitymatching.evaluation - 		5	|	57	|	2.46%
[INFO ] PyDI.entitymatching.evaluation - 		6	|	34	|	1.47%
[INFO ] PyDI.entitymatching.evaluation - 		7	|	39	|	1.68%
[INFO ] PyDI.entitymatching.evalua

## Part 4: Data Fusion

In [87]:
mbrainz["mbrainz_id"] = mbrainz["id"]

# Assign trust scores to datasets
mbrainz.attrs["trust_score"] = 3
discogs.attrs["trust_score"] = 1
lastfm.attrs["trust_score"] = 2

all_correspondences = pd.concat([correspondences_m2d, correspondences_m2l], ignore_index=True)
print(f'Total correspondences: {len(all_correspondences):,}')

Total correspondences: 6,851


## Step 1: Define Fusion Strategy 

In [88]:
from PyDI.fusion import DataFusionStrategy, longest_string, shortest_string, union, prefer_higher_trust, maximum

strategy = DataFusionStrategy('music_fusion_strategy')

strategy.add_attribute_fuser('name', shortest_string)
strategy.add_attribute_fuser('artist', longest_string)
strategy.add_attribute_fuser('release-date', prefer_higher_trust)
strategy.add_attribute_fuser('release-country', prefer_higher_trust)
strategy.add_attribute_fuser('duration', maximum)
strategy.add_attribute_fuser('tracks_track_name', union)
strategy.add_attribute_fuser('label', longest_string)

[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'name' using rule 'shortest_string'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'artist' using rule 'longest_string'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'release-date' using rule 'prefer_higher_trust'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'release-country' using rule 'prefer_higher_trust'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'duration' using rule 'maximum'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'tracks_track_name' using rule 'union'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'label' using rule 'longest_string'


## Step 2: Run Fusion

In [89]:
from PyDI.fusion import DataFusionEngine

engine = DataFusionEngine(strategy, debug=True, debug_format='json',debug_file=OUTPUT_DIR / "data_fusion" / "debug_fusion.jsonl")

fused = engine.run(
    datasets=[mbrainz, discogs, lastfm],
    correspondences=all_correspondences,
    id_column="id",
    include_singletons=False,
)
print(f'Fused rows: {len(fused):,}')
display(fused.head(5))

[INFO ] PyDI.fusion.engine - Fusion debug logging enabled; refer to /Users/aaronsteiner/Documents/GitHub/unsupervised-data-integration/usecases/input/music/output/music/data_fusion/debug_fusion.jsonl for detailed traces.
[INFO ] PyDI.fusion.engine - Starting data fusion with strategy 'music_fusion_strategy'
[INFO ] PyDI.fusion.engine - *    Loading correspondences    *
[INFO ] PyDI.fusion.engine - Correspondence ID coverage: matched 11409 of 11409 unique IDs
[INFO ] PyDI.fusion.engine - Created 30404 record groups from 6851 correspondences
[INFO ] PyDI.fusion.engine - Group Size Distribution of 30404 clusters:
[INFO ] PyDI.fusion.engine - 	Cluster Size	| Frequency	| Percentage
[INFO ] PyDI.fusion.engine - 	──────────────────────────────────────────────────
[INFO ] PyDI.fusion.engine - 		2	|	2265	|	7.45%
[INFO ] PyDI.fusion.engine - 		3	|	2293	|	7.54%
[INFO ] PyDI.fusion.engine - Attribute Consistencies:
[INFO ] PyDI.fusion.engine -     _id: 0.00
[INFO ] PyDI.fusion.engine -     artist:

Fused rows: 4,558


,_id,_fusion_sources,_fusion_source_datasets,name,name_longest_token,artist,id,release-country,duration,release-date,mbrainz_id,_fusion_confidence,_fusion_metadata,genre,label
0,lastFM_61508,"[lastFM_61508, mbrainz_25264]","[lastfm, mbrainz]",Won't Go,Won't,Brooklyn Dreams,lastFM_61508,United States,NaN,2013-04-16,mbrainz_25264,0.603571,"{'name_rule': 'shortest_string', 'name_sources...",NaN,NaN
1,discogs_27161,"[discogs_27161, lastFM_13089, mbrainz_6205]","[discogs, lastfm, mbrainz]",Angel Band: The Classic Mercury Recordings,Recordings,"Stanley Brothers, The",discogs_27161,United States,2191.0,1995-01-01,mbrainz_6205,0.724868,"{'name_rule': 'shortest_string', 'name_sources...","Folk, World, & Country",Mercury|PolyGram
2,mbrainz_169,"[mbrainz_169, discogs_452, lastFM_65381]","[mbrainz, discogs, lastfm]",Beyond EP,Beyond,"Cole, Dean",mbrainz_169,United Kingdom,NaN,2001-01-01,mbrainz_169,0.636508,"{'name_rule': 'shortest_string', 'name_sources...",Electronic,Corrosion Records
3,mbrainz_21896,"[mbrainz_21896, lastFM_6340]","[mbrainz, lastfm]",The Future,Future,"Future Roots, The",mbrainz_21896,Finland,1299.0,2010-01-01,mbrainz_21896,0.766282,"{'name_rule': 'shortest_string', 'name_sources...",NaN,NaN
4,discogs_62046,"[discogs_62046, lastFM_7879, mbrainz_7383]","[discogs, lastfm, mbrainz]",Going Nowhere,Nowhere,Dumptruck,discogs_62046,United Kingdom,1295.0,1987-01-01,mbrainz_7383,0.722222,"{'name_rule': 'shortest_string', 'name_sources...",Pop|Rock,Big Time


## Step 3: Evaluate Data Fusion

In [90]:
from PyDI.fusion import tokenized_match, year_only_match, set_equality_match, numeric_tolerance_match

strategy.add_evaluation_function("name", tokenized_match)
strategy.add_evaluation_function("artist", tokenized_match)
strategy.add_evaluation_function("duration", numeric_tolerance_match, tolerance=10) # 10 seconds tolerance
strategy.add_evaluation_function("release-date", year_only_match)
strategy.add_evaluation_function("release-country", tokenized_match)
strategy.add_evaluation_function("label", tokenized_match)
strategy.add_evaluation_function("tracks_track_name", set_equality_match)

[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'name'
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'artist'
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'duration' with params {'tolerance': 10}
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'release-date'
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'release-country'
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'label'
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'tracks_track_name'


In [91]:
from PyDI.fusion import DataFusionEvaluator
from PyDI.io import load_xml

fusion_test_set = load_xml(INPUT_DIR / 'fusion' / 'test_set.xml', name='fusion_test_set', nested_handling='aggregate')

# Create evaluator with our fusion strategy
evaluator = DataFusionEvaluator(strategy, debug=True, debug_file=OUTPUT_DIR / "data_fusion" / "debug_fusion_eval.jsonl", debug_format="json")

# Evaluate the fused results against the test set
print("Evaluating fusion results against test set...")
evaluation_results = evaluator.evaluate(
    fused_df=fused,
    fused_id_column='mbrainz_id',
    gold_df=fusion_test_set,
    gold_id_column='id',
)

# Display evaluation metrics
print("\nFusion Test Results:")
print("=" * 40)
for metric, value in evaluation_results.items():
    if isinstance(value, float):
        print(f"  {metric}: {value:.3f}")
    else:
        print(f"  {metric}: {value}")
        
print(f"\nOverall Accuracy: {evaluation_results.get('overall_accuracy', 0):.1%}")

[INFO ] PyDI.fusion.evaluation - Fusion evaluation debug logging enabled; refer to /Users/aaronsteiner/Documents/GitHub/unsupervised-data-integration/usecases/input/music/output/music/data_fusion/debug_fusion_eval.jsonl for mismatch details.
[INFO ] PyDI.fusion.evaluation - Starting fusion evaluation
[INFO ] PyDI.fusion.evaluation - Evaluation complete: 0.672 overall accuracy (88/131)
[INFO ] PyDI.fusion.evaluation - Evaluation mismatches by attribute (debug): 43 total
[INFO ] PyDI.fusion.evaluation - 	Attribute                        |  Errors | Percentage
[INFO ] PyDI.fusion.evaluation - 	───────────────────────────────────────────────────────
[INFO ] PyDI.fusion.evaluation - 	duration                         |      20 |     46.51%%
[INFO ] PyDI.fusion.evaluation - 	release-country                  |      12 |     27.91%%
[INFO ] PyDI.fusion.evaluation - 	artist                           |       4 |      9.30%%
[INFO ] PyDI.fusion.evaluation - 	name                             |     

Evaluating fusion results against test set...

Fusion Test Results:
  overall_accuracy: 0.672
  macro_accuracy: 0.679
  num_evaluated_records: 23
  num_evaluated_attributes: 6
  total_evaluations: 131
  total_correct: 88
  name_accuracy: 0.870
  name_count: 23
  artist_accuracy: 0.826
  artist_count: 23
  release-country_accuracy: 0.478
  release-country_count: 23
  duration_accuracy: 0.130
  duration_count: 23
  release-date_accuracy: 0.957
  release-date_count: 23
  label_accuracy: 0.812
  label_count: 16

Overall Accuracy: 67.2%


## Part 5: Reports and Metrics

In [92]:
# Source Overview Report
from PyDI.pipeline.reporting import save_source_overview

# Save source overview (using normalized datasets)
sources = {
    "mbrainz": mbrainz,
    "lastfm": lastfm,
    "discogs": discogs,
}

REPORTS_DIR = OUTPUT_DIR / "reporting"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

source_overview_path = save_source_overview(
    sources=sources,
    output_dir=REPORTS_DIR,
    target_columns=len(target_columns),
)
print(f"Source overview saved to: {source_overview_path}")
pd.read_csv(source_overview_path)

Source overview saved to: /Users/aaronsteiner/Documents/GitHub/unsupervised-data-integration/usecases/input/music/output/music/reporting/source_overview.csv


,Source,Rows,Columns,Data Density,Density in Target Schema (8 cols)
0,Mbrainz,4763,8,84.72%,84.72% (8/8 × 84.72%)
1,Lastfm,9865,5,89.40%,55.87% (5/8 × 89.40%)
2,Discogs,22627,8,98.43%,98.43% (8/8 × 98.43%)
3,Total Input,37255,8 (target),90.85% avg,85.41% weighted avg


In [93]:
# Schema Matching Report
from PyDI.pipeline.reporting import generate_column_mapping_table

SCHEMA_MATCHING_DIR = REPORTS_DIR / "schema_matching"
SCHEMA_MATCHING_DIR.mkdir(parents=True, exist_ok=True)

# Prepare mappings dictionary
mappings = {
    "mbrainz": mbrainz_mapping,
    "lastfm": lastfm_mapping,
    "discogs": discogs_mapping,
}

# Generate column mapping table
mapping_table = generate_column_mapping_table(mappings, target_schema)
mapping_table.to_csv(SCHEMA_MATCHING_DIR / "column_mapping.csv", index=False)
print(f"Schema matching report saved to: {SCHEMA_MATCHING_DIR}")
mapping_table

Schema matching report saved to: /Users/aaronsteiner/Documents/GitHub/unsupervised-data-integration/usecases/input/music/output/music/reporting/schema_matching


,Target Column,Discogs,Lastfm,Mbrainz
0,id,id,id,id
1,name,name,name,name
2,artist,artist,artist,artist
3,release-date,release-date,-,release-date
4,release-country,release-country,-,release-country
5,label,label,-,-
6,genre,genre,-,-
7,tracks,-,-,-
8,duration,-,duration,duration
9,Mapped,7/9,4/9,6/9


In [94]:
# Entity Matching Summary
MATCHING_DIR = REPORTS_DIR / "entity_matching"
MATCHING_DIR.mkdir(parents=True, exist_ok=True)

# Evaluate both entity matching pairs and save summaries
pairs_data = []

# MusicBrainz to Discogs
gt_test_m2d = load_csv(
    INPUT_DIR / "entitymatching" / "musicbrainz_2_discogs_test.csv", 
    name="test_entity_matching",
    header=None,
    names=['id1', 'id2', 'label'],
    add_index=False
)
eval_m2d = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_m2d,
    test_pairs=gt_test_m2d,
    out_dir=MATCHING_DIR
)
pairs_data.append({
    "left": "mbrainz",
    "right": "discogs", 
    "best_matcher": "RuleBasedMatcher",
    "f1": eval_m2d["f1"],
    "precision": eval_m2d["precision"],
    "recall": eval_m2d["recall"],
    "threshold": 0.9,
})

# MusicBrainz to LastFM
gt_test_m2l = load_csv(
    INPUT_DIR / "entitymatching" / "musicbrainz_2_lastfm_test.csv", 
    name="test_entity_matching",
    header=None,
    names=['id1', 'id2', 'label'],
    add_index=False
)
eval_m2l = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_m2l,
    test_pairs=gt_test_m2l,
    out_dir=MATCHING_DIR
)
pairs_data.append({
    "left": "mbrainz",
    "right": "lastfm",
    "best_matcher": "RuleBasedMatcher",
    "f1": eval_m2l["f1"],
    "precision": eval_m2l["precision"],
    "recall": eval_m2l["recall"],
    "threshold": 0.9,
})

# Save matching summary
matching_summary = pd.DataFrame(pairs_data)
matching_summary.to_csv(MATCHING_DIR / "matching_summary.csv", index=False)
print(f"Entity matching summary saved to: {MATCHING_DIR / 'matching_summary.csv'}")
matching_summary

[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  249
[INFO ] root -   True Negatives:  656
[INFO ] root -   False Positives: 11
[INFO ] root -   False Negatives: 84
[INFO ] root - Performance Metrics:
[INFO ] root -   Accuracy:  0.905
[INFO ] root -   Precision: 0.958
[INFO ] root -   Recall:    0.748
[INFO ] root -   F1-Score:  0.840
[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  228
[INFO ] root -   True Negatives:  655
[INFO ] root -   False Positives: 12
[INFO ] root -   False Negatives: 105
[INFO ] root - Performance Metrics:
[INFO ] root -   Accuracy:  0.883
[INFO ] root -   Precision: 0.950
[INFO ] root -   Recall:    0.685
[INFO ] root -   F1-Score:  0.796


Entity matching summary saved to: /Users/aaronsteiner/Documents/GitHub/unsupervised-data-integration/usecases/input/music/output/music/reporting/entity_matching/matching_summary.csv


,left,right,best_matcher,f1,precision,recall,threshold
0,mbrainz,discogs,RuleBasedMatcher,0.839798,0.957692,0.747748,0.9
1,mbrainz,lastfm,RuleBasedMatcher,0.795812,0.950000,0.684685,0.9


In [95]:
# Fusion Evaluation Summary
FUSION_DIR = REPORTS_DIR / "fusion"
FUSION_DIR.mkdir(parents=True, exist_ok=True)

# Save fusion comparison results (using evaluation_results from previous cell)
fusion_summary_data = [{
    "variant": "manual",
    "case": "heuristic",
    "test_accuracy": evaluation_results.get("overall_accuracy", 0),
    "test_correct": evaluation_results.get("total_correct", 0),
    "test_total": evaluation_results.get("total_evaluations", 0),
}]

fusion_summary = pd.DataFrame(fusion_summary_data)
fusion_summary.to_csv(FUSION_DIR / "fusion_comparison_summary.csv", index=False)
print(f"Fusion comparison saved to: {FUSION_DIR / 'fusion_comparison_summary.csv'}")

# Display per-attribute accuracy
per_attr_cols = [k for k in evaluation_results.keys() if k.endswith("_accuracy")]
per_attr_data = {k.replace("_accuracy", ""): [evaluation_results[k]] for k in per_attr_cols}
pd.DataFrame(per_attr_data)

Fusion comparison saved to: /Users/aaronsteiner/Documents/GitHub/unsupervised-data-integration/usecases/input/music/output/music/reporting/fusion/fusion_comparison_summary.csv


,overall,macro,name,artist,release-country,duration,release-date,label
0,0.671756,0.678895,0.869565,0.826087,0.478261,0.130435,0.956522,0.8125


In [96]:
# End-to-End Metrics
from PyDI.pipeline.end_to_end_metrics import EndToEndMetrics, SourceStats, calculate_density, save_end_to_end_report

# Calculate per-source statistics
per_source_stats = []
for name, df in sources.items():
    density = calculate_density(df)
    cols = [c for c in df.columns if not c.startswith("_")]
    per_source_stats.append(SourceStats(
        name=name,
        rows=len(df),
        columns=len(cols),
        density=density,
    ))

# Calculate merged records count (records from 2+ sources)
merged_count = 0
if "_fusion_source_datasets" in fused.columns:
    for val in fused["_fusion_source_datasets"]:
        if isinstance(val, list) and len(val) > 1:
            merged_count += 1
        elif isinstance(val, str) and val not in ("", "[]"):
            try:
                import json as json_module
                parsed = json_module.loads(val)
                if isinstance(parsed, list) and len(parsed) > 1:
                    merged_count += 1
            except:
                if len([s.strip() for s in val.split(",") if s.strip()]) > 1:
                    merged_count += 1

# Calculate metrics
total_input_rows = sum(s.rows for s in per_source_stats)
max_source = max(per_source_stats, key=lambda s: s.rows)
fused_density = calculate_density(fused)
fused_cols = [c for c in fused.columns if not c.startswith("_fusion_")]

# Get all unique columns across sources
all_columns = set()
for df in sources.values():
    all_columns.update(c for c in df.columns if not c.startswith("_"))

# Calculate average input density
total_non_null = sum(df[[c for c in df.columns if not c.startswith("_")]].notna().sum().sum() for df in sources.values())
total_cells = sum(len(df) * len(all_columns) for df in sources.values())
avg_input_density = total_non_null / total_cells if total_cells > 0 else 0.0

e2e_metrics = EndToEndMetrics(
    num_sources=len(sources),
    total_input_rows=total_input_rows,
    total_input_columns=len(all_columns),
    avg_input_density=avg_input_density,
    per_source_stats=per_source_stats,
    fused_rows=len(fused),
    fused_columns=len(fused_cols),
    fused_density=fused_density,
    max_source_rows=max_source.rows,
    row_gain_over_largest=len(fused) - max_source.rows,
    row_gain_pct=((len(fused) - max_source.rows) / max_source.rows * 100) if max_source.rows > 0 else 0.0,
    largest_source_name=max_source.name,
    largest_source_density=max_source.density,
    merged_records=merged_count,
    density_change=fused_density - avg_input_density,
)

# Save reports
txt_path, csv_path = save_end_to_end_report(e2e_metrics, REPORTS_DIR)
print(f"End-to-end report saved to: {csv_path}")

# Also save JSON format
import json
json_path = REPORTS_DIR / "end_to_end_metrics.json"
json_path.write_text(json.dumps(e2e_metrics.to_dict(), indent=2))

# Display the summary table
e2e_metrics.summary_table()

End-to-end report saved to: /Users/aaronsteiner/Documents/GitHub/unsupervised-data-integration/usecases/input/music/output/music/reporting/end_to_end_report.csv


,Metric,Value
0,Input Sources,3
1,Total Input Records,"37,255"
2,Fused Output Records,"4,558"
3,Fusion Ratio,87.8%
4,Row Gain Over Largest Source,"-18,069"
5,Row Gain Percentage,-79.9%
6,Merged Records,"4,558"
7,Input Columns (Target Schema),10
8,Fused Output Columns,11
9,Average Input Data Density,68.3%
